# Churn Prediction Model Training

This notebook demonstrates the training of a churn prediction model using RandomForestClassifier, optimized for recall to identify potential churning customers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score, roc_auc_score
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, precision_recall_curve
import pickle
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Data Loading and Exploration

In [ ]:
df = pd.read_csv('dataset_unificado.csv', sep=';')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")

In [ ]:
df.head()

## 2. Churn Definition and Target Variable Creation

We define churn based on the following criteria:
- Missing MRR (Monthly Recurring Revenue)
- Zero MRR
- Missing contract quantities
- Zero contract quantities
- Very recent customers (less than 30 days)

In [ ]:
df['CLIENTE_DESDE'] = pd.to_datetime(df['CLIENTE_DESDE'], errors='coerce')
df['dias_cliente'] = (pd.Timestamp.now() - df['CLIENTE_DESDE']).dt.days

df['churn'] = 0

churn_conditions = (
    (df['MRR_12M'].isna()) |
    (df['MRR_12M'] == 0) |
    (df['QTD_CONTRATACOES_12M'].isna()) |
    (df['QTD_CONTRATACOES_12M'] == 0) |
    (df['dias_cliente'] < 30)
)

df.loc[churn_conditions, 'churn'] = 1

print(f"Churn rate: {df['churn'].mean():.2%}")
print(f"Churn distribution:\n{df['churn'].value_counts()}")
print(f"\nChurn rate by customer age:")
print(df.groupby(pd.cut(df['dias_cliente'], bins=5))['churn'].mean())

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
df['churn'].value_counts().plot(kind='bar', color=['skyblue', 'salmon'])
plt.title('Churn Distribution')
plt.xlabel('Churn Status')
plt.ylabel('Count')
plt.xticks([0, 1], ['No Churn', 'Churn'])

plt.subplot(1, 2, 2)
df.groupby(pd.cut(df['dias_cliente'], bins=10))['churn'].mean().plot(kind='bar')
plt.title('Churn Rate by Customer Age')
plt.xlabel('Customer Age (days)')
plt.ylabel('Churn Rate')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 3. Feature Engineering

We create several engineered features to improve model performance:

In [ ]:
df_features = df.copy()

df_features['CLIENTE_DESDE'] = pd.to_datetime(df_features['CLIENTE_DESDE'], errors='coerce')
df_features['data_ultima_resposta_nps'] = pd.to_datetime(df_features['data_ultima_resposta_nps'], errors='coerce')

df_features['ano_cliente'] = df_features['CLIENTE_DESDE'].dt.year
df_features['mes_cliente'] = df_features['CLIENTE_DESDE'].dt.month
df_features['dias_cliente'] = (pd.Timestamp.now() - df_features['CLIENTE_DESDE']).dt.days

df_features['dias_ultima_resposta'] = (pd.Timestamp.now() - df_features['data_ultima_resposta_nps']).dt.days

df_features['MRR_12M'] = pd.to_numeric(df_features['MRR_12M'], errors='coerce')
df_features['QTD_CONTRATACOES_12M'] = pd.to_numeric(df_features['QTD_CONTRATACOES_12M'], errors='coerce')
df_features['VLR_CONTRATACOES_12M'] = pd.to_numeric(df_features['VLR_CONTRATACOES_12M'], errors='coerce')

df_features['resposta_NPS'] = pd.to_numeric(df_features['resposta_NPS'], errors='coerce')

nps_columns = [col for col in df_features.columns if 'Nota_' in col]
for col in nps_columns:
    df_features[col] = pd.to_numeric(df_features[col], errors='coerce')

df_features['media_notas_nps'] = df_features[nps_columns].mean(axis=1)
df_features['std_notas_nps'] = df_features[nps_columns].std(axis=1)
df_features['min_nota_nps'] = df_features[nps_columns].min(axis=1)
df_features['max_nota_nps'] = df_features[nps_columns].max(axis=1)

df_features['tem_nps'] = df_features['resposta_NPS'].notna().astype(int)
df_features['tem_contratacoes'] = df_features['QTD_CONTRATACOES_12M'].notna().astype(int)

print(f"Engineered features created. Total features: {len(df_features.columns)}")
print(f"New features: {[col for col in df_features.columns if col not in df.columns]}")

In [ ]:
feature_columns = [
    'ano_cliente', 'mes_cliente', 'dias_cliente', 'dias_ultima_resposta',
    'MRR_12M', 'QTD_CONTRATACOES_12M', 'VLR_CONTRATACOES_12M',
    'resposta_NPS', 'media_notas_nps', 'std_notas_nps', 'min_nota_nps', 'max_nota_nps',
    'tem_nps', 'tem_contratacoes'
]

feature_columns.extend(nps_columns)

X = df_features[feature_columns].copy()
y = df_features['churn']

print(f"Selected {len(feature_columns)} features")
print(f"Features: {feature_columns}")
print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

## 4. Data Preprocessing and Missing Value Handling

In [ ]:
print("Missing values before handling:")
print(X.isnull().sum())

X_clean = X.copy()

for col in X_clean.columns:
    if X_clean[col].dtype in ['int64', 'float64']:
        if X_clean[col].isnull().sum() > 0:
            median_val = X_clean[col].median()
            X_clean[col].fillna(median_val, inplace=True)
            print(f"Filled missing values in {col} with median: {median_val:.2f}")

print("\nMissing values after handling:")
print(X_clean.isnull().sum().sum())

## 5. Model Training with RandomForestClassifier

We use GridSearchCV to optimize hyperparameters, focusing on recall to identify churning customers.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Training set churn rate: {y_train.mean():.2%}")
print(f"Test set churn rate: {y_test.mean():.2%}")

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', 'balanced_subsample']
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    rf, param_grid, cv=5, scoring='recall', n_jobs=-1, verbose=1
)

print("Starting Grid Search for hyperparameter optimization...")
grid_search.fit(X_train_scaled, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation recall: {grid_search.best_score_:.4f}")

In [ ]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test_scaled)
y_pred_proba = best_model.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print("Model Performance:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"AUC: {auc:.4f}")

## 6. Model Evaluation and Visualization

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

## 7. Feature Importance Analysis

In [ ]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 Most Important Features:")
print(feature_importance.head(15))

## 8. Model Saving

In [ ]:
model_data = {
    'model': best_model,
    'scaler': scaler,
    'feature_names': feature_columns
}

with open('churn_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print("Model saved as churn_model.pkl")
print(f"Model includes: {list(model_data.keys())}")

## Summary

We have successfully trained a churn prediction model with the following characteristics:

**Model Type:** RandomForestClassifier
**Optimization Goal:** Recall (to identify as many churning customers as possible)
**Key Features:** Customer age, MRR, contract quantities, NPS scores, and derived metrics
**Performance Metrics:**
- Accuracy: [Value]
- Precision: [Value]
- Recall: [Value]
- AUC: [Value]

**Model Saved:** churn_model.pkl

The model is now ready for production use in identifying customers at risk of churning.